In [ ]:
!pip install kaggle --upgrade
!sudo apt-get install git-lfs=2.13.3
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!ls ~/.kaggle
#!kaggle competitions list
!kaggle competitions download -c leap-atmospheric-physics-ai-climsim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for kaggle: filename=kaggle-1.6.14-py3-none-any.whl size=105119 sha256=057492e5b08f1b0702473cb891b41eab05d8d19cccb3485612d5dc54ecf71aed
  Stored in directory: /root/.cache/pip/wheels/d7/54/06/8a8f40cb39536605feb9acaacd0237a95eba39e5065e6392f4
Successfully built kaggle
  Attempting uninstall: kaggle
    Found existing installation: kaggle 1.6.12
    Uninstalling kaggle-1.6.12:
      Successfully uninstalled kaggle-1.6.12
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package git-lfs is not available, but is referred to by another package.
This may mean that the package is missing, has been obsoleted, or
is only available from another source

E: Version '2.13.3' for 'git-lfs' was not found


Saving kaggle.json to kaggle.json
kaggle.json
100% 72.5G/72.5G [41:20<00:00, 28.5MB/s]
100% 72.5G/72.5G [41:20<00:00, 31.4MB/s]


In [ ]:
# prompt: código para subir o arquivo .zip para meu repositório git

!git clone https://github.com/gabriel-ac6/LEAP-dataset.git
!mv leap-atmospheric-physics-ai-climsim.zip LEAP-dataset/
!cd LEAP-dataset
!git add leap-atmospheric-physics-ai-climsim.zip
!git commit -m "Added Leap Atmospheric Physics AI ClimSim data"
!git push


Detected operating system as Ubuntu/jammy.
Checking for curl...
Detected curl...
Checking for gpg...
Detected gpg...
Detected apt version as 2.4.12
Running apt-get update... done.
Installing apt-transport-https... done.
Installing /etc/apt/sources.list.d/github_git-lfs.list...done.
Importing packagecloud gpg key... Packagecloud gpg key imported to /etc/apt/keyrings/github_git-lfs-archive-keyring.gpg
done.
Running apt-get update... done.

The repository is setup! You can now install packages.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.5.1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
Cloning into 'LEAP-dataset'...
/content/LEAP-dataset/LEAP-dataset/LEAP-dataset/LEAP-dataset/LEAP-dataset/LEAP-dataset
Updated Git hooks.
Git LFS initialized.
Tracking "/content/leap-atmospheric-physics-ai-climsim.zip"
error: remote origin already exists.
[main (root-commit) c88ff11] Adicionando

In [ ]:
import pandas as pd
import numpy as np
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from sklearn.metrics import mean_squared_error

# Função para aplicar a ponderação
def apply_weighting(predictions, inverse_std_devs):
    weighted_predictions = predictions.copy()

    # Ignorar certos níveis (zero out)
    for var in ['ptend_q0001', 'ptend_q0002', 'ptend_q0003', 'ptend_u', 'ptend_v']:
        for level in range(12):
            col = f'{var}_{level}'
            if col in weighted_predictions:
                weighted_predictions[col] = 0.0
    for level in range(12, 15):
        col = 'ptend_q0002_' + str(level)
        if col in weighted_predictions:
            weighted_predictions[col] = 0.0

    # Aplicar o inverso do desvio padrão
    for col in weighted_predictions.columns:
        if col in inverse_std_devs:
            weighted_predictions[col] *= inverse_std_devs[col]

    return weighted_predictions

# Ler os dados do arquivo zip
with zipfile.ZipFile('/content/leap-atmospheric-physics-ai-climsim.zip', 'r') as z:
    with z.open('train.csv') as f:
        train_df = pd.read_csv(f)
    with z.open('test.csv') as f:
        test_df = pd.read_csv(f)

# Definir colunas de entrada e saída
input_columns = [
    'state_t', 'state_q0001', 'state_q0002', 'state_q0003', 'state_u', 'state_v', 'state_ps',
    'pbuf_SOLIN', 'pbuf_LHFLX', 'pbuf_SHFLX', 'pbuf_TAUX', 'pbuf_TAUY', 'pbuf_COSZRS',
    'cam_in_ALDIF', 'cam_in_ALDIR', 'cam_in_ASDIF', 'cam_in_ASDIR', 'cam_in_LWUP',
    'cam_in_ICEFRAC', 'cam_in_LANDFRAC', 'cam_in_OCNFRAC', 'cam_in_SNOWHLAND',
    'pbuf_ozone', 'pbuf_CH4', 'pbuf_N2O'
]

# Planificar variáveis com níveis verticais (0-59)
input_columns_expanded = []
for col in input_columns:
    if col in ['state_t', 'state_q0001', 'state_q0002', 'state_q0003', 'state_u', 'state_v', 'pbuf_ozone', 'pbuf_CH4', 'pbuf_N2O']:
        input_columns_expanded.extend([f'{col}_{i}' for i in range(60)])
    else:
        input_columns_expanded.append(col)

target_columns = [
    'ptend_t', 'ptend_q0001', 'ptend_q0002', 'ptend_q0003', 'ptend_u', 'ptend_v',
    'cam_out_NETSW', 'cam_out_FLWDS', 'cam_out_PRECSC', 'cam_out_PRECC',
    'cam_out_SOLS', 'cam_out_SOLL', 'cam_out_SOLSD', 'cam_out_SOLLD'
]

target_columns_expanded = []
for col in target_columns:
    if col in ['ptend_t', 'ptend_q0001', 'ptend_q0002', 'ptend_q0003', 'ptend_u', 'ptend_v']:
        target_columns_expanded.extend([f'{col}_{i}' for i in range(60)])
    else:
        target_columns_expanded.append(col)

# Dividir em conjuntos de treino e validação
X = train_df[input_columns_expanded]
y = train_df[target_columns_expanded]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar os dados
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)

# Calcular o desvio padrão das variáveis de saída
std_devs = y_train.std()
inverse_std_devs = 1 / std_devs

# Função para treinar e avaliar o modelo
def train_and_evaluate_model(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled):
    model = Sequential([
        Dense(512, input_shape=(X_train_scaled.shape[1],)),
        LeakyReLU(alpha=0.01),
        Dropout(0.2),
        Dense(256),
        LeakyReLU(alpha=0.01),
        Dropout(0.2),
        Dense(128),
        LeakyReLU(alpha=0.01),
        Dense(y_train_scaled.shape[1], activation='linear')
    ])

    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    model.fit(X_train_scaled, y_train_scaled, epochs=50, batch_size=32, validation_data=(X_val_scaled, y_val_scaled))

    y_val_pred_scaled = model.predict(X_val_scaled)
    y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled)

    # Aplicar ponderação nas previsões
    y_val_pred_weighted = apply_weighting(pd.DataFrame(y_val_pred, columns=target_columns_expanded), inverse_std_devs)

    # Calcular a métrica de avaliação ponderada
    mse_weighted = mean_squared_error(y_val, y_val_pred_weighted)
    print(f'Validation Weighted MSE: {mse_weighted}')

    return model

# Treinar e avaliar o modelo
model = train_and_evaluate_model(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled)

# Prever no conjunto de teste
X_test_scaled = scaler_X.transform(test_df[input_columns_expanded])
y_test_pred_scaled = model.predict(X_test_scaled)
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled)

# Aplicar ponderação nas previsões de teste
y_test_pred_weighted = apply_weighting(pd.DataFrame(y_test_pred, columns=target_columns_expanded), inverse_std_devs)

# Salvar as previsões ponderadas
y_test_pred_weighted.to_csv('submission.csv', index=False)
